# DSPy: Compound Systems for RAG, Agents and Classification using LLMS

In [ ]:
# - using public models as teachers 
# - distill small local "student" models
# e.g. 

In [16]:
import dspy
import random
from typing import Literal
from dspy.datasets import DataLoader
from datasets import load_dataset
from sklearn.model_selection import train_test_split

In [1]:
lm = dspy.LM('anthropic/claude-3-7-sonnet-20250219')
dspy.configure(lm=lm)

### Basic Example

In [4]:
class Classify(dspy.Signature):
    """Classify the sentiment of a given sentence."""
    sentence: str = dspy.InputField()
    sentiment: Literal['positive', 'negative'] = dspy.OutputField()
    confidence: float = dspy.OutputField()

classify = dspy.Predict(Classify)
classify(sentence="This book was super fun to read, though not the last chapter.")

Prediction(
    sentiment='positive',
    confidence=0.85
)

### Agents

In [7]:
def evaluate_math(expression: str):
    return dspy.PythonInterpreter({}).execute(expression)

def search_wikipedia(query: str):
    results = dspy.ColBERTv2(url='http://20.102.90.50:2017/wiki17_abstracts')(query, k=3)
    return [x['text'] for x in results]

react = dspy.ReAct("question -> answer: float", tools=[evaluate_math, search_wikipedia])
pred = react(question = "What is the distance to alpha centauri divided by one astronomical unit?")

In [8]:
pred

Prediction(
    trajectory={'thought_0': 'I need to find the distance to Alpha Centauri and then divide it by one astronomical unit (AU). Let me first search for information about Alpha Centauri to find its distance from Earth.', 'tool_name_0': 'search_wikipedia', 'tool_args_0': {'query': 'Alpha Centauri distance from Earth'}, 'observation_0': ['Alpha Centauri Bb | Alpha Centauri Bb (α Cen B b) was a proposed exoplanet orbiting the K-type main-sequence star Alpha Centauri B, located 4.37 light-years from Earth in the southern constellation of Centaurus. Its existence was announced in October 2012 by a team of European observers, and the finding received widespread media attention. However, the announcement was met with scepticism by some astronomers, who thought that the European team was over-interpreting its data.', 'Alpha Centauri Bc | Alpha Centauri Bc is a suspected exoplanet orbiting Alpha Centauri B, part of Alpha Centauri system, the closest system to our own. It is located abo

### Classification

We use a CSV file as input and create individual `Example`s that can be used to train a model
```python
dspy.Example(
    text="input text",
    hint=row['type_of_output'],
    output_type=row['type_of_output']
).with_inputs("text", "hint")
```

In [ ]:
# Define classification signature for output types
class CGIARClassify(dspy.Signature):
    """Classify text according to type of output (Methods, Solutions, Impacts, or Other)."""
    text: str = dspy.InputField()
    output_type: Literal['methods', 'solutions', 'impacts', 'other'] = dspy.OutputField()

# Read the data
df = pd.read_csv('priv/ice/2022-climate-results-classified.csv')

# Split into training (50 samples) and validation sets
train_df, val_df = train_test_split(df, train_size=50, random_state=42)

# Create training examples with hints
trainset = []
for _, row in train_df.iterrows():
    for col in ['title', 'description', 'abstract']:
        trainset.append(
            dspy.Example(
                text=row[col],
                hint=row['type_of_output'],
                output_type=row['type_of_output']
            ).with_inputs("text", "hint")
        )
random.Random(0).shuffle(trainset)

# Define the DSPy module for classification
classify = dspy.ChainOfThoughtWithHint(CGIARClassify)

# Optimize via BootstrapFinetune
optimizer = dspy.BootstrapFinetune(
    metric=(lambda x, y, trace=None: x.output_type == y.output_type),
    num_threads=24
)
optimized_classifier = optimizer.compile(classify, trainset=trainset)

# Validate on remaining data
for col in ['title', 'description', 'abstract']:
    val_df[f'{col}_output_type'] = val_df[col].apply(lambda x: optimized_classifier(text=x).output_type)

# Save validation results
val_df.to_csv('output_type_validation.csv', index=False)
